ich möchte eine auswertung über alle verspätungen, station = 'Leipzig Hbf'. für die verspätung in min sollen alle möglichen verpassten anschlusszüge ausgewertet werden. hinsichtlich der nächtsmöglichen verbindung, oder ob alternativen über andere strecken möglich sind. dazu müssten die spalte 'path' hinzugezogen werden, um mögliche schnittpunkte der haltestellen zu analysieren. für den fall das die verbindungen nicht ausreichend sind, sollen neue fiktive erstellt werden. diese sollen sowohl bundesweit wie auch regional sein. für die reginalen verbindung stelle ich daten von www.mdv.de zur verfügung.

In [1]:

import pandas as pd
import numpy as np
import datetime as dt
import random
import os
#import matplotlib.pyplot as plt

from datetime import datetime, timedelta


from core.data import load_from_kaggle

/home/ulfgar/Lehrgänge/Projekte/portfolio/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_link = "nokkyu/deutsche-bahn-db-delays" # replace with your dataset link from Kaggle 
destination = "../data/raw"
dataset_name = dataset_link.split("/")[-1]

files = load_from_kaggle(
    dataset_link=dataset_link, 
    destination=destination,
    )

Destination directory '../data/raw/deutsche-bahn-db-delays' already exists with files. Skipping download (replace=False).


In [3]:
df = pd.read_csv("/".join(["../data/raw/", dataset_name, files[0]]))
df.head()

,ID,line,path,eva_nr,category,station,state,city,zip,long,lat,arrival_plan,departure_plan,arrival_change,departure_change,arrival_delay_m,departure_delay_m,info,arrival_delay_check,departure_delay_check
0,1573967790757085557-2407072312-14,20,Stolberg(Rheinl)Hbf Gl.44|Eschweiler-St.Jöris|...,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,50.767800,2024-07-08 00:00:00,2024-07-08 00:01:00,2024-07-08 00:03:00,2024-07-08 00:04:00,3,3,NaN,on_time,on_time
1,349781417030375472-2407080017-1,18,NaN,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,50.767800,NaN,2024-07-08 00:17:00,NaN,NaN,0,0,NaN,on_time,on_time
2,7157250219775883918-2407072120-25,1,Hamm(Westf)Hbf|Kamen|Kamen-Methler|Dortmund-Ku...,8000406,4,Aachen-Rothe Erde,Nordrhein-Westfalen,Aachen,52066,6.116475,50.770202,2024-07-08 00:03:00,2024-07-08 00:04:00,2024-07-08 00:03:00,2024-07-08 00:04:00,0,0,NaN,on_time,on_time
3,349781417030375472-2407080017-2,18,Aachen Hbf,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,50.780360,2024-07-08 00:20:00,2024-07-08 00:21:00,NaN,NaN,0,0,NaN,on_time,on_time
4,1983158592123451570-2407080010-3,33,Herzogenrath|Kohlscheid,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,50.780360,2024-07-08 00:20:00,2024-07-08 00:21:00,2024-07-08 00:20:00,2024-07-08 00:21:00,0,0,NaN,on_time,on_time


In [4]:

#  Nur Daten für Leipzig Hbf
df_leipzig = df[df["station"] == "Leipzig Hbf"].copy()
df_leipzig.head()

,ID,line,path,eva_nr,category,station,state,city,zip,long,lat,arrival_plan,departure_plan,arrival_change,departure_change,arrival_delay_m,departure_delay_m,info,arrival_delay_check,departure_delay_check
3411,8575858415775127007-2407080012-1,113,NaN,8010205,1,Leipzig Hbf,Sachsen,Leipzig,4109,12.382064,51.345471,NaN,2024-07-08 00:12:00,NaN,2024-07-08 00:12:00,0,0,NaN,on_time,on_time
3412,6865594399444333693-2407080009-1,20,NaN,8010205,1,Leipzig Hbf,Sachsen,Leipzig,4109,12.382064,51.345471,NaN,2024-07-08 00:09:00,NaN,2024-07-08 00:09:00,0,0,NaN,on_time,on_time
3413,-5680384712273970736-2407080013-1,5X,NaN,8010205,1,Leipzig Hbf,Sachsen,Leipzig,4109,12.382064,51.345471,NaN,2024-07-08 00:13:00,NaN,2024-07-08 00:13:00,0,0,NaN,on_time,on_time
13266,6603576928487697059-2407080449-1,2,NaN,8010205,1,Leipzig Hbf,Sachsen,Leipzig,4109,12.382064,51.345471,NaN,2024-07-08 04:49:00,NaN,NaN,0,0,NaN,on_time,on_time
13267,9018503409677789560-2407080412-1,6,NaN,8010205,1,Leipzig Hbf,Sachsen,Leipzig,4109,12.382064,51.345471,NaN,2024-07-08 04:12:00,NaN,NaN,0,0,NaN,on_time,on_time


In [5]:
df.shape

(2061357, 20)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2061357 entries, 0 to 2061356
Data columns (total 20 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ID                     object 
 1   line                   object 
 2   path                   object 
 3   eva_nr                 int64  
 4   category               int64  
 5   station                object 
 6   state                  object 
 7   city                   object 
 8   zip                    int64  
 9   long                   float64
 10  lat                    float64
 11  arrival_plan           object 
 12  departure_plan         object 
 13  arrival_change         object 
 14  departure_change       object 
 15  arrival_delay_m        int64  
 16  departure_delay_m      int64  
 17  info                   object 
 18  arrival_delay_check    object 
 19  departure_delay_check  object 
dtypes: float64(2), int64(5), object(13)
memory usage: 314.5+ MB


In [7]:
# =====================================================
# PHASE 1.5: Fiktive MDV- und bundesweite Verbindungen erstellen
# =====================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# -----------------------------
# PARAMETER
# -----------------------------
n_bundesweit = 30
n_regional = 60
datum = datetime(2024, 7, 8)  # Bezugstag

np.random.seed(42)  # für Reproduzierbarkeit

# -----------------------------
# 1️⃣ Bundesweite fiktive Züge
# -----------------------------
# Zufällige Abfahrtszeiten
abfahrts_bundes = [datum + timedelta(hours=np.random.randint(0,24),
                                     minutes=np.random.randint(0,60))
                   for _ in range(n_bundesweit)]

# Zufällige Wartezeiten (1–15 min)
warte_bundes = np.random.randint(1,16, size=n_bundesweit)

# DataFrame
fiktiv_bundes = pd.DataFrame({
    "ursprungs_abfahrt": abfahrts_bundes,
    "best_trip_id": range(1000, 1000+n_bundesweit),
    #"best_departure": [t + timedelta(minutes=w) for t,w in zip(abfahrts_bundes, warte_bundes)],
    "best_departure": [t + timedelta(minutes=int(w)) for t,w in zip(abfahrts_bundes, warte_bundes)],
    "wartezeit_min": warte_bundes,
    "verspaetung_min": 0,
    "quelle": "MDV"
})

# -----------------------------
# 2️⃣ Regionale fiktive Züge (MDV)
# -----------------------------
abfahrts_regional = [datum + timedelta(hours=np.random.randint(0,24),
                                      minutes=np.random.randint(0,60))
                     for _ in range(n_regional)]

warte_regional = np.random.randint(1,16, size=n_regional)

fiktiv_regional = pd.DataFrame({
    "ursprungs_abfahrt": abfahrts_regional,
    "best_trip_id": range(2000, 2000+n_regional),
    #"best_departure": [t + timedelta(minutes=w) for t,w in zip(abfahrts_regional, warte_regional)],
    "best_departure": [t + timedelta(minutes=int(w)) for t,w in zip(abfahrts_regional, warte_regional)],
    "wartezeit_min": warte_regional,
    "verspaetung_min": 0,
    "quelle": "MDV"
})

# -----------------------------
# 3️⃣ Zusammenführen der fiktiven Daten
# -----------------------------
fiktive_verbindungen = pd.concat([fiktiv_bundes, fiktiv_regional], ignore_index=True)

# -----------------------------
# Ergebnis prüfen
# -----------------------------
print(f"Anzahl fiktiver Verbindungen: {len(fiktive_verbindungen)}")
fiktive_verbindungen.head()


Anzahl fiktiver Verbindungen: 90


,ursprungs_abfahrt,best_trip_id,best_departure,wartezeit_min,verspaetung_min,quelle
0,2024-07-08 06:51:00,1000,2024-07-08 06:55:00,4,0,MDV
1,2024-07-08 14:42:00,1001,2024-07-08 14:54:00,12,0,MDV
2,2024-07-08 07:20:00,1002,2024-07-08 07:35:00,15,0,MDV
3,2024-07-08 06:57:00,1003,2024-07-08 07:09:00,12,0,MDV
4,2024-07-08 18:22:00,1004,2024-07-08 18:29:00,7,0,MDV


In [8]:
#NICHT AUSFÜHREN
#print("Neue Gesamtanzahl Datensätze:", len(df_leipzig_all))
#print(df_leipzig_all.tail(5)[["line","departure_plan","departure_delay_m"]])


In [9]:
#  Spalten für Datum/Zeit in Datetime-Format umwandeln
df_leipzig["arrival_plan"] = pd.to_datetime(df_leipzig["arrival_plan"], errors="coerce")
df_leipzig["departure_plan"] = pd.to_datetime(df_leipzig["departure_plan"], errors="coerce")


In [10]:
#  Nur Einträge für den 8. Juli 2024
date_filter = (df_leipzig["arrival_plan"].dt.date == datetime(2024, 7, 8).date()) | \
              (df_leipzig["departure_plan"].dt.date == datetime(2024, 7, 8).date())

df_leipzig = df_leipzig[date_filter].copy()

In [11]:
#  Überblick verschaffen
print("Anzahl der Datensätze für Leipzig Hbf am 2024-07-08:", len(df_leipzig))
print(df_leipzig[["line", "arrival_plan", "departure_plan", "arrival_delay_m", "departure_delay_m"]].head())

# 6️⃣ Fehlende Werte prüfen
print("\nFehlende Werte pro Spalte:")
print(df_leipzig.isnull().sum())

Anzahl der Datensätze für Leipzig Hbf am 2024-07-08: 174
      line arrival_plan      departure_plan  arrival_delay_m  \
3411   113          NaT 2024-07-08 00:12:00                0   
3412    20          NaT 2024-07-08 00:09:00                0   
3413    5X          NaT 2024-07-08 00:13:00                0   
13266    2          NaT 2024-07-08 04:49:00                0   
13267    6          NaT 2024-07-08 04:12:00                0   

       departure_delay_m  
3411                   0  
3412                   0  
3413                   0  
13266                  0  
13267                  0  

Fehlende Werte pro Spalte:
ID                         0
line                       0
path                     174
eva_nr                     0
category                   0
station                    0
state                      0
city                       0
zip                        0
long                       0
lat                        0
arrival_plan             174
departure_plan      

In [12]:
# Prüfen, ob es überhaupt Verspätungen > 0 gibt
print("Züge mit Ankunftsverspätung > 0:", (df_leipzig["arrival_delay_m"] > 0).sum())
print("Züge mit Abfahrtsverspätung > 0:", (df_leipzig["departure_delay_m"] > 0).sum())


Züge mit Ankunftsverspätung > 0: 0
Züge mit Abfahrtsverspätung > 0: 23


verspätungen auswerten

In [13]:
# Nur verspätete Abfahrten auswählen
delayed = df_leipzig[df_leipzig["departure_delay_m"] > 0]

In [14]:
# Basis-Kennzahlen berechnen
print("Anzahl verspäteter Abfahrten:", len(delayed))
print("Durchschnittliche Abfahrtsverspätung (min):", delayed["departure_delay_m"].mean())
print("Maximale Abfahrtsverspätung (min):", delayed["departure_delay_m"].max())

Anzahl verspäteter Abfahrten: 23
Durchschnittliche Abfahrtsverspätung (min): 1.9565217391304348
Maximale Abfahrtsverspätung (min): 7


In [15]:
#  Häufigkeitsverteilung ansehen
print("\nHäufigkeit der Verspätungsminuten:")
print(delayed["departure_delay_m"].value_counts().sort_index())


Häufigkeit der Verspätungsminuten:
departure_delay_m
1    16
2     2
3     1
4     1
5     1
6     1
7     1
Name: count, dtype: int64


In [16]:
# Übersichtstabelle mit Kennzahlen
delay_summary = pd.DataFrame({
    "Kennzahl": ["Anzahl verspäteter Abfahrten", "Durchschnitt (min)", "Maximum (min)", "Minimum (min)"],
    "Wert": [
        len(delayed),
        delayed["departure_delay_m"].mean(),
        delayed["departure_delay_m"].max(),
        delayed["departure_delay_m"].min()
    ]
})

print(delay_summary)


                       Kennzahl       Wert
0  Anzahl verspäteter Abfahrten  23.000000
1            Durchschnitt (min)   1.956522
2                 Maximum (min)   7.000000
3                 Minimum (min)   1.000000


Anschlusszüge auswerten

Welche Züge nach einer Abfahrt von Leipzig Hbf folgen (mögliche Anschlussverbindungen).
Ob eine Verspätung dazu führt, dass ein Anschluss nicht mehr erreichbar ist. (ggf. Alternativen MDV)

mindestumstiegszeit 5 min?

In [17]:

# Nur verspätete Abfahrten
delayed = df_leipzig[df_leipzig["departure_delay_m"] > 0].copy()

# Umwandeln in datetime, falls noch nicht geschehen
delayed["departure_plan"] = pd.to_datetime(delayed["departure_plan"], errors="coerce")
delayed["departure_change"] = pd.to_datetime(delayed["departure_change"], errors="coerce")

# Sortieren nach geplanter Abfahrt
delayed = delayed.sort_values("departure_plan")

print(delayed[["line", "departure_plan", "departure_delay_m"]].head())


      line      departure_plan  departure_delay_m
22789   13 2024-07-08 05:02:00                  1
55622   50 2024-07-08 07:00:00                  1
73126  110 2024-07-08 08:06:00                  1
89506   50 2024-07-08 09:00:00                  1
89507  110 2024-07-08 09:06:00                  1


In [18]:
# Alle Züge (auch pünktliche) nach Abfahrtszeit sortieren
all_trains = df_leipzig.copy()
all_trains["departure_plan"] = pd.to_datetime(all_trains["departure_plan"], errors="coerce")
all_trains = all_trains.sort_values("departure_plan")

# Für jeden verspäteten Zug: nächster geplanter Zug nach seiner Abfahrt
anschluesse = []

for _, row in delayed.iterrows():
    abfahrt = row["departure_plan"]
    verspaetung = row["departure_delay_m"]
    effektive_abfahrt = abfahrt + pd.to_timedelta(verspaetung, unit="m")
    
    # Finde nächste Abfahrt, die nach dieser effektiven Abfahrt liegt
    next_train = all_trains[all_trains["departure_plan"] > effektive_abfahrt].head(1)
    
    if not next_train.empty:
        anschluss = {
            "verspäteter_Zug": row["line"],
            "Abfahrt_geplant": abfahrt,
            "Abfahrt_effektiv": effektive_abfahrt,
            "Verspätung_min": verspaetung,
            "Anschluss_Zug": next_train.iloc[0]["line"],
            "Anschluss_Abfahrt": next_train.iloc[0]["departure_plan"],
            "Zeitdifferenz_min": (next_train.iloc[0]["departure_plan"] - effektive_abfahrt).total_seconds() / 60
        }
        anschluesse.append(anschluss)

anschluss_df = pd.DataFrame(anschluesse)
anschluss_df.head()


,verspäteter_Zug,Abfahrt_geplant,Abfahrt_effektiv,Verspätung_min,Anschluss_Zug,Anschluss_Abfahrt,Zeitdifferenz_min
0,13,2024-07-08 05:02:00,2024-07-08 05:03:00,1,110,2024-07-08 05:06:00,3.0
1,50,2024-07-08 07:00:00,2024-07-08 07:01:00,1,13,2024-07-08 07:04:00,3.0
2,110,2024-07-08 08:06:00,2024-07-08 08:07:00,1,20,2024-07-08 08:09:00,2.0
3,50,2024-07-08 09:00:00,2024-07-08 09:01:00,1,13,2024-07-08 09:04:00,3.0
4,110,2024-07-08 09:06:00,2024-07-08 09:07:00,1,20,2024-07-08 09:09:00,2.0


In [19]:
MIN_UMSTIEGSZEIT = 5  # Minuten

anschluss_df["Anschluss_verpasst"] = anschluss_df["Zeitdifferenz_min"] < MIN_UMSTIEGSZEIT

# Ergebnisübersicht
print(anschluss_df[["verspäteter_Zug", "Verspätung_min", "Anschluss_Zug", 
                    "Zeitdifferenz_min", "Anschluss_verpasst"]])


   verspäteter_Zug  Verspätung_min Anschluss_Zug  Zeitdifferenz_min  \
0               13               1           110                3.0   
1               50               1            13                3.0   
2              110               1            20                2.0   
3               50               1            13                3.0   
4              110               1            20                2.0   
5               50               1            13                3.0   
6               50               1            13                3.0   
7               13               1           110                1.0   
8              110               1            20                2.0   
9               13               1           110                1.0   
10              50               4           110                2.0   
11              13               1           110                1.0   
12               6               3            10                4.0   
13    

MDV-Daten einlesen

In [20]:
# =====================================================
# 🧭 PHASE 4: GTFS-DATEN LADEN UND VORBEREITEN
# =====================================================

import pandas as pd
import os

mdv_path = "../data/raw/"

# 1️⃣ Alle wichtigen GTFS-Dateien einlesen
stops = pd.read_csv(os.path.join(mdv_path, "stops.txt"))
stop_times = pd.read_csv(os.path.join(mdv_path, "stop_times.txt"))
trips = pd.read_csv(os.path.join(mdv_path, "trips.txt"))
routes = pd.read_csv(os.path.join(mdv_path, "routes.txt"))
calendar = pd.read_csv(os.path.join(mdv_path, "calendar.txt"))
calendar_dates = pd.read_csv(os.path.join(mdv_path, "calendar_dates.txt"))

print("✅ GTFS-Dateien erfolgreich geladen.")
print(f"Stops: {len(stops)}, Stop_times: {len(stop_times)}, Trips: {len(trips)}, Routes: {len(routes)}")

# 2️⃣ Kurzer Überblick über die wichtigsten Spalten
print("\nBeispiel: stops.txt")
display(stops.head(3))

print("\nBeispiel: routes.txt")
display(routes.head(3))

print("\nBeispiel: stop_times.txt")
display(stop_times.head(3))


✅ GTFS-Dateien erfolgreich geladen.
Stops: 5299, Stop_times: 1583706, Trips: 79863, Routes: 604

Beispiel: stops.txt


,stop_id,stop_name,stop_lat,stop_lon
0,145,"Leipzig, Arcus Park",51.361549,12.446576
1,152,"Leipzig, Bergstr.",51.340535,12.402015
2,154,"Leipzig, Breitscheidhof",51.351846,12.286913



Beispiel: routes.txt


,route_id,agency_id,route_short_name,route_long_name,route_type
0,800413RB113,800413,RB113,Regionalbahn RB113,2
1,800445RB76,800445,RB76,Regionalbahn RB76,2
2,800445RB78,800445,RB78,Regionalbahn RB78,2



Beispiel: stop_times.txt


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type
0,1,00:12:00,00:12:00,8010205,1,0,0
1,1,00:17:00,00:18:00,8010208,2,0,0
2,1,00:19:00,00:20:00,8011495,3,0,0


In [21]:
# =====================================================
# 🚉 PHASE 4 – TEIL 2: VERKNÜPFUNG UND ANSCHLUSSPRÜFUNG
# =====================================================

import pandas as pd
from datetime import datetime, timedelta

# 1️⃣ Nur verspätete Abfahrten von Leipzig Hbf betrachten
delayed_leipzig = df_leipzig[df_leipzig["departure_delay_m"] > 0].copy()

# 2️⃣ GTFS-Stop-ID für Leipzig Hbf finden
leipzig_stop_ids = stops[stops["stop_name"].str.contains("Leipzig Hbf", case=False, na=False)]["stop_id"].unique()
print("GTFS-Stop-IDs für Leipzig Hbf:", leipzig_stop_ids)

# 3️⃣ Alle GTFS-Abfahrten, die in Leipzig Hbf stattfinden
gtfs_leipzig_departures = stop_times[stop_times["stop_id"].isin(leipzig_stop_ids)].copy()

# 4️⃣ Zeitfelder konvertieren (GTFS-Zeiten sind Strings wie "12:34:00")
def to_datetime_gtfs(t):
    try:
        return datetime(2024, 7, 8, int(t.split(":")[0]) % 24, int(t.split(":")[1]))
    except:
        return None

gtfs_leipzig_departures["departure_time_dt"] = gtfs_leipzig_departures["departure_time"].apply(to_datetime_gtfs)

# 5️⃣ Beispiel: Anschlussprüfung für die ersten 5 verspäteten Abfahrten
anschluesse = []

for _, row in delayed_leipzig.head(5).iterrows():
    abfahrt = row["departure_plan"]
    zug = row["line"]
    delay = row["departure_delay_m"]
    neue_abfahrt = abfahrt + timedelta(minutes=delay)
    
    # Finde GTFS-Abfahrten in Leipzig Hbf innerhalb der nächsten 30 Minuten
    zeitfenster = (gtfs_leipzig_departures["departure_time_dt"] >= neue_abfahrt) & \
                  (gtfs_leipzig_departures["departure_time_dt"] <= neue_abfahrt + timedelta(minutes=30))
    
    moegliche_anschluesse = gtfs_leipzig_departures[zeitfenster].copy()
    moegliche_anschluesse["ursprungs_zug"] = zug
    moegliche_anschluesse["ursprungs_abfahrt"] = abfahrt
    moegliche_anschluesse["verspaetung_min"] = delay
    
    anschluesse.append(moegliche_anschluesse)

anschluss_df = pd.concat(anschluesse, ignore_index=True)

print(f"Beispielhafte Anschlussprüfung durchgeführt ({len(anschluss_df)} mögliche Anschlüsse gefunden).")
display(anschluss_df.head(10))


GTFS-Stop-IDs für Leipzig Hbf: [8010205 8098205]
Beispielhafte Anschlussprüfung durchgeführt (338 mögliche Anschlüsse gefunden).


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,departure_time_dt,ursprungs_zug,ursprungs_abfahrt,verspaetung_min
0,574,05:11:00,05:13:00,8098205,7,0,0,2024-07-08 05:13:00,13,2024-07-08 05:02:00,1
1,672,05:18:00,05:18:00,8098205,11,0,0,2024-07-08 05:18:00,13,2024-07-08 05:02:00,1
2,710,05:18:00,05:20:00,8098205,11,0,0,2024-07-08 05:20:00,13,2024-07-08 05:02:00,1
3,727,05:11:00,05:13:00,8098205,7,0,0,2024-07-08 05:13:00,13,2024-07-08 05:02:00,1
4,797,05:18:00,05:18:00,8098205,11,0,0,2024-07-08 05:18:00,13,2024-07-08 05:02:00,1
5,806,05:18:00,05:20:00,8098205,11,0,0,2024-07-08 05:20:00,13,2024-07-08 05:02:00,1
6,907,05:28:00,05:30:00,8098205,13,0,0,2024-07-08 05:30:00,13,2024-07-08 05:02:00,1
7,921,05:28:00,05:30:00,8098205,19,0,0,2024-07-08 05:30:00,13,2024-07-08 05:02:00,1
8,922,05:28:00,05:30:00,8098205,13,0,0,2024-07-08 05:30:00,13,2024-07-08 05:02:00,1
9,923,05:28:00,05:30:00,8098205,13,0,0,2024-07-08 05:30:00,13,2024-07-08 05:02:00,1


In [22]:
# =====================================================
# 🚦 PHASE 5: ANSCHLUSSANALYSE UND BEWERTUNG
# =====================================================

import pandas as pd
from datetime import timedelta

# 1️⃣ Parameter: Mindestumsteigezeit (realistisch: 5 Minuten)
MIN_UMSTEIGEZEIT = timedelta(minutes=5)

# 2️⃣ Wir prüfen: Wurde der Anschluss erreicht oder verpasst?
anschluss_df["zeitdifferenz"] = anschluss_df["departure_time_dt"] - anschluss_df["ursprungs_abfahrt"]
anschluss_df["anschluss_erreicht"] = anschluss_df["zeitdifferenz"] > MIN_UMSTEIGEZEIT

# 3️⃣ Statistische Übersicht
gesamt = len(anschluss_df)
erreicht = anschluss_df["anschluss_erreicht"].sum()
verpasst = gesamt - erreicht

print(f"Gesamtzahl möglicher Anschlüsse: {gesamt}")
print(f"➡️  Erreichte Anschlüsse: {erreicht}")
print(f"❌ Verpasste Anschlüsse: {verpasst}")

# 4️⃣ Anteil in Prozent
anschluss_stats = pd.DataFrame({
    "Kategorie": ["Erreicht", "Verpasst"],
    "Anzahl": [erreicht, verpasst],
    "Prozent": [erreicht/gesamt*100, verpasst/gesamt*100]
})

display(anschluss_stats)

# 5️⃣ Liste der verpassten Anschlüsse (für spätere Alternativsuche)
verpasste_anschluesse = anschluss_df[~anschluss_df["anschluss_erreicht"]].copy()

print("\nBeispielhafte verpasste Anschlüsse:")
display(verpasste_anschluesse.head(10))


Gesamtzahl möglicher Anschlüsse: 338
➡️  Erreichte Anschlüsse: 272
❌ Verpasste Anschlüsse: 66


,Kategorie,Anzahl,Prozent
0,Erreicht,272,80.473373
1,Verpasst,66,19.526627



Beispielhafte verpasste Anschlüsse:


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,departure_time_dt,ursprungs_zug,ursprungs_abfahrt,verspaetung_min,zeitdifferenz,anschluss_erreicht
10,1075,05:01:00,05:03:00,8098205,18,0,0,2024-07-08 05:03:00,13,2024-07-08 05:02:00,1,0 days 00:01:00,False
11,1076,05:01:00,05:03:00,8098205,18,0,0,2024-07-08 05:03:00,13,2024-07-08 05:02:00,1,0 days 00:01:00,False
12,1077,05:01:00,05:03:00,8098205,7,0,0,2024-07-08 05:03:00,13,2024-07-08 05:02:00,1,0 days 00:01:00,False
13,1078,05:01:00,05:03:00,8098205,7,0,0,2024-07-08 05:03:00,13,2024-07-08 05:02:00,1,0 days 00:01:00,False
20,1305,05:01:00,05:03:00,8098205,18,0,0,2024-07-08 05:03:00,13,2024-07-08 05:02:00,1,0 days 00:01:00,False
21,1306,05:01:00,05:03:00,8098205,7,0,0,2024-07-08 05:03:00,13,2024-07-08 05:02:00,1,0 days 00:01:00,False
24,1367,05:03:00,05:04:00,8098205,10,0,0,2024-07-08 05:04:00,13,2024-07-08 05:02:00,1,0 days 00:02:00,False
28,1460,05:03:00,05:04:00,8098205,10,0,0,2024-07-08 05:04:00,13,2024-07-08 05:02:00,1,0 days 00:02:00,False
53,6820,05:06:00,05:06:00,8010205,1,2,3,2024-07-08 05:06:00,13,2024-07-08 05:02:00,1,0 days 00:04:00,False
55,6900,05:06:00,05:06:00,8010205,1,2,3,2024-07-08 05:06:00,13,2024-07-08 05:02:00,1,0 days 00:04:00,False


In [30]:


# DB-DataFrame (verpasste Anschlüsse) vorbereiten
#verpasste_anschluesse["trip_id"] = verpasste_anschluesse["ID"]
verpasste_anschluesse["trip_id"] = range(len(verpasste_anschluesse))


# Fiktive MDV-Alternativen
fiktive_verbindungen = fiktive_verbindungen.rename(columns={"best_trip_id": "trip_id"})
alternative_df = fiktive_verbindungen.copy()
alternative_df["departure_time_dt"] = alternative_df["best_departure"]



MIN_UMSTEIGEZEIT	Pufferzeit, die ein Fahrgast zum Umsteigen braucht.
zeitdifferenz	Zeit zwischen der verspäteten Ankunft und der nächsten Abfahrt.
anschluss_erreicht	True, wenn die Zeitdifferenz größer als 5 min ist.
anschluss_stats	Zeigt den Anteil erreichter und verpasster Anschlüsse.
verpasste_anschluesse	Liste für spätere Alternativrouten-Suche.

Gesamtzahl möglicher Anschlüsse: 338
➡️  Erreichte Anschlüsse: 290
❌ Verpasste Anschlüsse: 48


In [ ]:
#Komplettcode Phse 6 + 7

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# -----------------------------
# 1️⃣ Fiktive MDV-Verbindungen erstellen
# -----------------------------
datum = datetime(2024,7,8)
n_bundesweit = 20
n_regional = 50

# Bundesweite Verbindungen
abfahrts_bundes = [datum + timedelta(hours=np.random.randint(0,24),
                                     minutes=np.random.randint(0,60))
                   for _ in range(n_bundesweit)]
warte_bundes = np.random.randint(1,16, size=n_bundesweit)  # Wartezeiten 1-15 min

fiktiv_bundes = pd.DataFrame({
    "ursprungs_abfahrt": abfahrts_bundes,
    "trip_id": range(1000, 1000+n_bundesweit),
    "best_departure": [t + timedelta(minutes=int(w)) for t,w in zip(abfahrts_bundes, warte_bundes)],
    "wartezeit_min": warte_bundes,
    "verspaetung_min": 0,
    "quelle": "MDV"
})

# Regionale Verbindungen
abfahrts_regional = [datum + timedelta(hours=np.random.randint(0,24),
                                       minutes=np.random.randint(0,60))
                     for _ in range(n_regional)]
warte_regional = np.random.randint(1,16, size=n_regional)

fiktiv_regional = pd.DataFrame({
    "ursprungs_abfahrt": abfahrts_regional,
    "trip_id": range(2000, 2000+n_regional),
    "best_departure": [t + timedelta(minutes=int(w)) for t,w in zip(abfahrts_regional, warte_regional)],
    "wartezeit_min": warte_regional,
    "verspaetung_min": 0,
    "quelle": "MDV"
})

# Alle MDV-Daten zusammen
fiktive_verbindungen = pd.concat([fiktiv_bundes, fiktiv_regional], ignore_index=True)
print(f"Anzahl fiktiver Verbindungen: {len(fiktive_verbindungen)}")

# -----------------------------
# 2️⃣ Alternative DataFrame für Phase 6 vorbereiten
# -----------------------------
alternative_df = fiktive_verbindungen.copy()
alternative_df["departure_time_dt"] = alternative_df["best_departure"]

# -----------------------------
# 3️⃣ DB-Verpasste Anschlüsse vorbereiten
# -----------------------------
# Beispiel: verpasste_anschluesse aus vorheriger Phase 6
# Wir erzeugen hier eine eindeutige trip_id, falls noch nicht vorhanden
verpasste_anschluesse["trip_id"] = range(len(verpasste_anschluesse))
verpasste_anschluesse["verspaetung_min"] = verpasste_anschluesse.get("departure_delay_m", 0)

# -----------------------------
# 4️⃣ Phase 6: Beste Alternativen finden
# -----------------------------
ALT_WINDOW = timedelta(minutes=60)         # Zeitfenster nach ursprünglicher Abfahrt
MIN_UMSTIEGEZEIT = timedelta(minutes=1)   # minimale Umstiegszeit

beste_alternativen = []

for idx, v in verpasste_anschluesse.iterrows():
    ursprungs_abfahrt = v["ursprungs_abfahrt"]

    # Kandidaten: alle MDV-Alternativen innerhalb Zeitfenster
    cands = alternative_df[
        (alternative_df["departure_time_dt"] >= ursprungs_abfahrt + MIN_UMSTIEGEZEIT) &
        (alternative_df["departure_time_dt"] <= ursprungs_abfahrt + ALT_WINDOW)
    ].copy()

    if cands.empty:
        # keine Alternative gefunden
        beste_alternativen.append({
            "ursprungs_abfahrt": ursprungs_abfahrt,
            "trip_id": v["trip_id"],
            "best_departure": None,
            "wartezeit_min": None,
            "verspaetung_min": v["verspaetung_min"],
            "quelle": "MDV"
        })
    else:
        # die früheste Alternative nehmen
        next_trip = cands.sort_values("departure_time_dt").iloc[0]
        wartezeit = (next_trip["departure_time_dt"] - ursprungs_abfahrt).total_seconds() / 60.0
        beste_alternativen.append({
            "ursprungs_abfahrt": ursprungs_abfahrt,
            "trip_id": next_trip["trip_id"],
            "best_departure": next_trip["departure_time_dt"],
            "wartezeit_min": wartezeit,
            "verspaetung_min": v["verspaetung_min"],
            "quelle": "MDV"
        })

beste_alternativen = pd.DataFrame(beste_alternativen)

# -----------------------------
# 5️⃣ Tageszeit-Kategorisierung
# -----------------------------
def zeitabschnitt(dt):
    h = dt.hour
    if 0 <= h < 6: return "Nacht (0–6)"
    if 6 <= h < 10: return "Früh (6–10)"
    if 10 <= h < 14: return "Mittags (10–14)"
    if 14 <= h < 18: return "Nachmittags (14–18)"
    if 18 <= h < 22: return "Abends (18–22)"
    return "Spät (22–24)"

beste_alternativen["tageszeit"] = beste_alternativen["ursprungs_abfahrt"].apply(zeitabschnitt)

db_df = verpasste_anschluesse[["ursprungs_abfahrt","verspaetung_min"]].copy()
db_df["wartezeit_min"] = 0
db_df["quelle"] = "DB"
db_df["tageszeit"] = db_df["ursprungs_abfahrt"].apply(zeitabschnitt)

# -----------------------------
# 6️⃣ Zusammenführen und Auswertung
# -----------------------------
vergleich_df = pd.concat([db_df, beste_alternativen[["ursprungs_abfahrt","wartezeit_min","verspaetung_min","quelle","tageszeit"]]], ignore_index=True)

vergleich_summary = vergleich_df.groupby("quelle").agg({
    "verspaetung_min":["mean","max"],
    "wartezeit_min":["mean","max"]
}).round(2)

vergleich_summary_time = vergleich_df.groupby(["quelle","tageszeit"]).agg({
    "verspaetung_min":["mean","max"],
    "wartezeit_min":["mean","max"]
}).round(2)

print("✅ Gesamtvergleich DB vs. MDV:")
print(vergleich_summary)
print("✅ Vergleich nach Tageszeit:")
print(vergleich_summary_time)


Anzahl fiktiver Verbindungen: 70
✅ Gesamtvergleich DB vs. MDV:
       verspaetung_min     wartezeit_min      
                  mean max          mean   max
quelle                                        
DB                 0.0   0          0.00   0.0
MDV                0.0   0         22.85  33.0
✅ Vergleich nach Tageszeit:
                   verspaetung_min     wartezeit_min      
                              mean max          mean   max
quelle tageszeit                                          
DB     Früh (6–10)             0.0   0          0.00   0.0
       Nacht (0–6)             0.0   0          0.00   0.0
MDV    Früh (6–10)             0.0   0         25.14  33.0
       Nacht (0–6)             0.0   0         10.00  10.0


In [33]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# -----------------------------
# 1️⃣ Fiktive MDV-Daten (einfach) erstellen
# -----------------------------
datum = datetime(2024,7,8)
n_bundesweit = 20
n_regional = 50

def create_fictive(n, start_id):
    abfahrts = [datum + timedelta(hours=np.random.randint(0,24),
                                 minutes=np.random.randint(0,60))
                for _ in range(n)]
    warte = np.random.randint(1,16, size=n)
    df = pd.DataFrame({
        "ursprungs_abfahrt": abfahrts,
        "trip_id": range(start_id, start_id+n),
        "best_departure": [t + timedelta(minutes=int(w)) for t,w in zip(abfahrts, warte)],
        "wartezeit_min": warte,
        "verspaetung_min": 0,
        "quelle": "MDV"
    })
    return df

mdv_bundes = create_fictive(n_bundesweit, 1000)
mdv_regional = create_fictive(n_regional, 2000)
fiktive_verbindungen = pd.concat([mdv_bundes, mdv_regional], ignore_index=True)

# -----------------------------
# 2️⃣ DB-Daten vorbereiten
# -----------------------------
verpasste_anschluesse["trip_id"] = range(len(verpasste_anschluesse))
verpasste_anschluesse["verspaetung_min"] = verpasste_anschluesse.get("departure_delay_m", 0)
verpasste_anschluesse["wartezeit_min"] = 0
verpasste_anschluesse["quelle"] = "DB"

# -----------------------------
# 3️⃣ Tageszeit-Spalte
# -----------------------------
def zeitabschnitt(dt):
    h = dt.hour
    if 0 <= h < 6: return "Nacht (0–6)"
    if 6 <= h < 10: return "Früh (6–10)"
    if 10 <= h < 14: return "Mittags (10–14)"
    if 14 <= h < 18: return "Nachmittags (14–18)"
    if 18 <= h < 22: return "Abends (18–22)"
    return "Spät (22–24)"

verpasste_anschluesse["tageszeit"] = verpasste_anschluesse["ursprungs_abfahrt"].apply(zeitabschnitt)
fiktive_verbindungen["tageszeit"] = fiktive_verbindungen["ursprungs_abfahrt"].apply(zeitabschnitt)

# -----------------------------
# 4️⃣ DB + MDV zusammenführen
# -----------------------------
vergleich_df = pd.concat([verpasste_anschluesse[["ursprungs_abfahrt","verspaetung_min","wartezeit_min","quelle","tageszeit"]],
                          fiktive_verbindungen[["ursprungs_abfahrt","verspaetung_min","wartezeit_min","quelle","tageszeit"]]],
                         ignore_index=True)

# -----------------------------
# 5️⃣ Tabellarische Auswertung
# -----------------------------
# Gesamtvergleich
vergleich_summary = vergleich_df.groupby("quelle").agg({
    "verspaetung_min":["mean","max"],
    "wartezeit_min":["mean","max"]
}).round(2)

# Vergleich nach Tageszeit
vergleich_summary_time = vergleich_df.groupby(["quelle","tageszeit"]).agg({
    "verspaetung_min":["mean","max"],
    "wartezeit_min":["mean","max"]
}).round(2)

print("✅ Gesamtvergleich DB vs. MDV:")
print(vergleich_summary)
print("\n✅ Vergleich nach Tageszeit:")
print(vergleich_summary_time)


✅ Gesamtvergleich DB vs. MDV:
       verspaetung_min     wartezeit_min    
                  mean max          mean max
quelle                                      
DB                 0.0   0          0.00   0
MDV                0.0   0          7.57  15

✅ Vergleich nach Tageszeit:
                           verspaetung_min     wartezeit_min    
                                      mean max          mean max
quelle tageszeit                                                
DB     Früh (6–10)                     0.0   0          0.00   0
       Nacht (0–6)                     0.0   0          0.00   0
MDV    Abends (18–22)                  0.0   0          7.46  14
       Früh (6–10)                     0.0   0          7.00  13
       Mittags (10–14)                 0.0   0          8.55  14
       Nachmittags (14–18)             0.0   0          5.25  13
       Nacht (0–6)                     0.0   0          7.35  15
       Spät (22–24)                    0.0   0         10.83  15


In [34]:
# -----------------------------
# 6️⃣ Erweiterte Stunden-Analyse
# -----------------------------
# Stunde aus Abfahrtszeit extrahieren
vergleich_df["stunde"] = vergleich_df["ursprungs_abfahrt"].dt.hour

# Gruppieren nach Quelle und Stunde
stunden_summary = vergleich_df.groupby(["quelle","stunde"]).agg(
    anzahl_anschluesse = ("ursprungs_abfahrt", "count"),
    avg_verspaetung = ("verspaetung_min", "mean"),
    max_verspaetung = ("verspaetung_min", "max"),
    avg_wartezeit = ("wartezeit_min", "mean"),
    max_wartezeit = ("wartezeit_min", "max")
).round(2)

# Optional: Tabelle nach Stunden sortieren
stunden_summary = stunden_summary.reset_index().sort_values(["quelle","stunde"])

print("✅ Stundenweise Auswertung DB vs. MDV:")
print(stunden_summary)


✅ Stundenweise Auswertung DB vs. MDV:
   quelle  stunde  anzahl_anschluesse  avg_verspaetung  max_verspaetung  \
0      DB       5                  10              0.0                0   
1      DB       7                  16              0.0                0   
2      DB       8                  12              0.0                0   
3      DB       9                  28              0.0                0   
4     MDV       0                   4              0.0                0   
5     MDV       1                   2              0.0                0   
6     MDV       2                   3              0.0                0   
7     MDV       3                   3              0.0                0   
8     MDV       4                  10              0.0                0   
9     MDV       5                   1              0.0                0   
10    MDV       6                   1              0.0                0   
11    MDV       7                   2              0.0        

In [35]:
# -----------------------------
# 7️⃣ Pivot-Tabelle: DB vs. MDV stundenweise
# -----------------------------
pivot = vergleich_df.pivot_table(
    index="stunde",
    columns="quelle",
    values=["verspaetung_min","wartezeit_min"],
    aggfunc=["mean","max"]
).round(2)

# Spalten übersichtlicher umbenennen
pivot.columns = ['_'.join(col).strip() for col in pivot.columns.values]
pivot = pivot.reset_index()

print("✅ Pivot-Tabelle DB vs. MDV stundenweise:")
print(pivot)


✅ Pivot-Tabelle DB vs. MDV stundenweise:
    stunde  mean_verspaetung_min_DB  mean_verspaetung_min_MDV  \
0        0                      NaN                       0.0   
1        1                      NaN                       0.0   
2        2                      NaN                       0.0   
3        3                      NaN                       0.0   
4        4                      NaN                       0.0   
5        5                      0.0                       0.0   
6        6                      NaN                       0.0   
7        7                      0.0                       0.0   
8        8                      0.0                       0.0   
9        9                      0.0                       0.0   
10      11                      NaN                       0.0   
11      12                      NaN                       0.0   
12      13                      NaN                       0.0   
13      14                      NaN              